In [0]:
%pip install kagglehub

In [0]:
dbutils.library.restartPython()

In [0]:
# --- dev bootstrap: torna o módulo importável em serverless interativo ---
import sys, os

nb_dir = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
# .../files/notebooks/tables/bronze  ->  .../files/src
src_path = os.path.abspath(os.path.join("/Workspace" + nb_dir, "../../../src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [0]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

from olist_lakehouse.transformations import add_metadata

os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get("kaggle", "KAGGLE_API_TOKEN")

In [0]:
KAGGLE_OLIST_DATASETS = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]

dbutils.widgets.text(
    "datasets",
    defaultValue=",".join(KAGGLE_OLIST_DATASETS),
    label="OLIST datasets, separated by comma."
)
datasets = dbutils.widgets.get("datasets")

In [0]:
for dataset in datasets.split(","):
    pandas_df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "olistbr/brazilian-ecommerce",
        dataset,
    )

    spark_df = spark.createDataFrame(pandas_df)

    
    spark_df.write.mode("overwrite").saveAsTable()

In [0]:
"""
olistbr/brazilian-ecommerce is a static dataset from Kaggle,
in this case I use a simple download so later I am able to extract
and load the files into Unity Catalog

This requires a Kaggle API Token, defined in 'kaggle' databricks secrets scope:
databricks secrets create-scope kaggle
databricks secrets put-secret kaggle KAGGLE_API_TOKEN --string-value "<your_token>"
"""